# Topic modeling (LDA) — exploratory pass

Explores what policy topics are discussed in state-parliament speeches, scoped to each state's
legislative period immediately before vs. after AfD entry. This does not measure morality or
politeness — the topic assigned to each speech is meant as a **control** variable for later
analysis (some topics may be more polarized/moralized than others, independent of AfD entry).

Runs two independent topic-count-selection methods and compares them: gensim LDA scored by c_v
coherence, and sklearn LDA scored by held-out log-likelihood (the latter per the practical guide
https://medium.com/data-science/practical-guide-to-topic-modeling-with-lda-05cd6b027bdf, which
argues coherence is unreliable for tuning -- rather than pick a side, both run and get compared).

See `docs/superpowers/specs/2026-08-14-topic-modeling-lda-design.md` (local-only, not tracked in
git) for full design rationale.

In [1]:
import os
import sys

import spacy
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath(".."))
load_dotenv("../.env")

from topic_modeling_lib import (
    build_gensim_corpus,
    build_sklearn_corpus,
    gensim_coherence_scan,
    load_corpus,
    make_spacy_preprocessor,
    sklearn_loglikelihood_search,
    timed,
)

DATA_ROOT = os.environ["DATA_ROOT"]
timing_log = []

## Parameters

Start small (single state, sampled) to get a fast timing read before scaling up -- see the
"Timing" cell at the end for what that read implies about a full run.

In [2]:
STATES = None       # None = all 16 states
PRE_POST = None        # "pre", "post", or None for both
SAMPLE_N = None         # None = use every speech in scope
SAMPLE_SEED = 42        # random_state for load_corpus's sampling
K_RANGE = [10, 15, 20, 25, 30, 35, 40, 45, 50]
SKLEARN_N_ITER = 9      # None = try every k in K_RANGE

# Cache key inputs for everything downstream. SAMPLE_SEED is in here because a different
# random sample of the same size is a different corpus, not a cache hit. "preprocessing" is
# filled in from the actual loaded spaCy model in the next cell, so it can never drift from
# the model that produced the tokens.
run_params = {
    "states": STATES,
    "pre_post": PRE_POST,
    "sample_n": SAMPLE_N,
    "sample_seed": SAMPLE_SEED,
}

In [3]:
with timed("load_corpus", log=timing_log):
    corpus_df = load_corpus(
        DATA_ROOT, states=STATES, pre_post=PRE_POST, sample_n=SAMPLE_N, seed=SAMPLE_SEED,
    )
print(f"{len(corpus_df):,} documents loaded")

with timed("load spaCy model", log=timing_log):
    nlp = spacy.load("de_core_news_lg")
run_params["preprocessing"] = f"{nlp.meta['lang']}_{nlp.meta['name']}_{nlp.meta['version']}"
print(f"preprocessing: {run_params['preprocessing']}")
preprocess = make_spacy_preprocessor(nlp)

with timed("preprocess", log=timing_log):
    tokenized_docs = preprocess(corpus_df["text"].tolist())
print(f"Example tokens: {tokenized_docs[0][:10]}")

[load_corpus] 1.78s
444,189 documents loaded


[load spaCy model] 0.62s
preprocessing: de_core_news_lg_3.8.0


[preprocess] 9916.08s
Example tokens: ['herr', 'präsident', 'geehrt', 'dame', 'herr', 'mitteilen', 'ministerin', 'jutta', 'lieske', 'rücktritt']


## Vectorize and fit both models across K

Both methods share the same `tokenized_docs` and `K_RANGE`, but vectorize independently (gensim's
`Dictionary`/BoW vs. sklearn's `CountVectorizer`/doc-term matrix) since each library needs its own
input format. Results are cached under `measurement/run_history/topic_modeling/`, keyed by
`run_params` + K range + method + model seed -- rerunning this notebook with the same parameters
loads from disk instead of re-fitting.

In [4]:
with timed("build_gensim_corpus", log=timing_log):
    dictionary, gensim_corpus = build_gensim_corpus(tokenized_docs)

with timed("gensim_coherence_scan (all k)", log=timing_log):
    gensim_results = gensim_coherence_scan(
        tokenized_docs, dictionary, gensim_corpus, k_range=K_RANGE, params=run_params,
    )
gensim_results[["k", "coherence", "seconds"]]

[build_gensim_corpus] 43.32s


[gensim k=10] 144.25s, coherence=0.3749


[gensim k=15] 192.67s, coherence=0.4358


[gensim k=20] 235.94s, coherence=0.4339


[gensim k=25] 288.02s, coherence=0.4612


...

[gensim k=40] 419.41s, coherence=0.5120


[gensim k=45] 441.47s, coherence=0.4699


[gensim k=50] 465.17s, coherence=0.4662


[gensim_coherence_scan (all k)] 2872.41s


,k,coherence,seconds
0,10,0.374894,144.250016
1,15,0.435796,192.670595
2,20,0.433917,235.936681
3,25,0.461157,288.024784
4,30,0.470636,314.207407
5,35,0.478727,369.743590
6,40,0.511983,419.414830
7,45,0.469932,441.474179
8,50,0.466152,465.171280


In [5]:
with timed("build_sklearn_corpus", log=timing_log):
    vectorizer, dtm = build_sklearn_corpus(tokenized_docs)

with timed("sklearn_loglikelihood_search (all k)", log=timing_log):
    sklearn_results = sklearn_loglikelihood_search(
        dtm, k_range=K_RANGE, params=run_params, n_iter=SKLEARN_N_ITER,
    )
sklearn_results[["k", "log_likelihood", "seconds"]]

[build_sklearn_corpus] 10.85s


[sklearn k=45] 964.42s, log_likelihood=-127973411.07


[sklearn k=50] 979.55s, log_likelihood=-128676987.61


[sklearn k=20] 767.62s, log_likelihood=-124504449.01


[sklearn k=30] 829.94s, log_likelihood=-125817607.05


...

[sklearn k=25] 796.65s, log_likelihood=-125169522.19


[sklearn k=10] 706.32s, log_likelihood=-123392844.30


[sklearn k=15] 778.99s, log_likelihood=-124056317.74


[sklearn_loglikelihood_search (all k)] 7641.78s


,k,log_likelihood,seconds
0,45,-1.279734e+08,964.419181
1,50,-1.286770e+08,979.551855
2,20,-1.245044e+08,767.620142
3,30,-1.258176e+08,829.935178
4,40,-1.273062e+08,924.856717
5,35,-1.266039e+08,890.816549
6,25,-1.251695e+08,796.651592
7,10,-1.233928e+08,706.322313
8,15,-1.240563e+08,778.985385


## Compare the two methods' preferred K

In [6]:
best_gensim_k = int(gensim_results.loc[gensim_results["coherence"].idxmax(), "k"])
best_sklearn_k = int(sklearn_results.loc[sklearn_results["log_likelihood"].idxmax(), "k"])
print(f"gensim (c_v coherence) prefers k={best_gensim_k}")
print(f"sklearn (log-likelihood) prefers k={best_sklearn_k}")
print("Agreement" if best_gensim_k == best_sklearn_k else "Disagreement -- inspect both before picking K")

gensim (c_v coherence) prefers k=40
sklearn (log-likelihood) prefers k=10
Disagreement -- inspect both before picking K


## Inspect topics for the chosen K

In [7]:
chosen_k = best_gensim_k  # change after inspecting the comparison above
chosen_model = gensim_results.set_index("k").loc[chosen_k, "model"]

for topic_id, terms in chosen_model.print_topics(num_words=10):
    print(f"Topic {topic_id}: {terms}\n")

Topic 2: 0.067*"verein" + 0.062*"sport" + 0.060*"stiftung" + 0.051*"ehrenamtlich" + 0.039*"feuerwehr" + 0.028*"engagement" + 0.024*"ehrenamt" + 0.020*"verband" + 0.020*"freiwillig" + 0.014*"engagieren"

Topic 31: 0.023*"arbeit" + 0.019*"beschäftigter" + 0.015*"unternehmen" + 0.012*"mindestlohn" + 0.012*"arbeitnehmer" + 0.011*"sozial" + 0.011*"mensch" + 0.010*"wirtschaft" + 0.009*"arbeiten" + 0.009*"lohn"

Topic 37: 0.020*"thüringer" + 0.011*"thüringen" + 0.011*"regelung" + 0.007*"fall" + 0.006*"artikel" + 0.006*"rechtlich" + 0.006*"entscheidung" + 0.006*"grund" + 0.005*"gelten" + 0.005*"verfassung"

Topic 1: 0.017*"verfahren" + 0.013*"fall" + 0.012*"justiz" + 0.012*"enden" + 0.011*"monat" + 0.009*"richter" + 0.009*"gericht" + 0.008*"herr" + 0.008*"asylbewerber" + 0.007*"ten"

Topic 4: 0.042*"sagen" + 0.041*"mal" + 0.016*"wissen" + 0.014*"glauben" + 0.013*"brauchen" + 0.012*"einfach" + 0.011*"problem" + 0.010*"reden" + 0.010*"finden" + 0.009*"ding"

Topic 0: 0.126*"hochschule" + 0.035*"

## Timing summary — for estimating full-corpus runtime

This ran on the sample size and state(s) set in the Parameters cell above. Multiply the
preprocess/vectorize/fit seconds-per-document by the full pre/post-AfD corpus size (see
`preprocessing/afd_period_window.py`'s printed document count) to estimate a full run's cost
before committing to it.

In [8]:
import pandas as pd

timing_df = pd.DataFrame(timing_log)
timing_df["seconds_per_doc"] = timing_df["seconds"] / len(corpus_df)
timing_df

,step,seconds,seconds_per_doc
0,load_corpus,1.781661,0.000004
1,load spaCy model,0.622280,0.000001
2,preprocess,9916.076076,0.022324
3,build_gensim_corpus,43.316359,0.000098
4,gensim_coherence_scan (all k),2872.409561,0.006467
5,build_sklearn_corpus,10.850611,0.000024
6,sklearn_loglikelihood_search (all k),7641.784738,0.017204
